# Reading an Unsupervised Result Honestly

**Sessions 15–18 · no homework depends on this, the capstone often does**

Unsupervised methods always return something. PCA returns components, k-means
returns clusters, t-SNE returns a picture — on structured data and on noise
alike, with no error message to tell the difference. This notebook is about the
checks that separate the two.

Run every cell; the outputs are real runs.

In [1]:
import numpy as np
from sklearn.datasets import load_digits, make_blobs
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)
digits = load_digits()
X = StandardScaler().fit_transform(digits.data)
y = digits.target
print("digits:", X.shape, " classes:", len(np.unique(y)))

digits: (1797, 64)  classes: 10


## 1. PCA: how many components, and what they cost

`explained_variance_ratio_` is the honest summary. Read the **cumulative** curve
and pick a threshold before you look at any downstream score.

In [2]:
from sklearn.decomposition import PCA

pca = PCA().fit(X)
cum = np.cumsum(pca.explained_variance_ratio_)
for target in (0.80, 0.90, 0.95, 0.99):
    k = int(np.searchsorted(cum, target)) + 1
    print(f"{target:.0%} of variance needs {k:>2} of {X.shape[1]} components")

80% of variance needs 21 of 64 components
90% of variance needs 31 of 64 components
95% of variance needs 40 of 64 components
99% of variance needs 54 of 64 components


Sixty-four pixels compress to a couple of dozen numbers with almost nothing
lost — which is the whole argument for dimensionality reduction, stated in one
table.

**Scaling first is not optional.** PCA maximises variance, so a feature measured
in larger units dominates the components for no reason but its units:

In [3]:
raw = digits.data.copy()
loudest = int(np.argmax(raw.var(axis=0)))   # pick a pixel that actually varies
raw[:, loudest] = raw[:, loudest] * 1000    # ...and measure it in different units
print(f"rescaled pixel #{loudest}")

first_unscaled = PCA(n_components=1).fit(raw).explained_variance_ratio_[0]
first_scaled = PCA(n_components=1).fit(StandardScaler().fit_transform(raw)).explained_variance_ratio_[0]
print(f"first component, unscaled: {first_unscaled:.4f} of variance")
print(f"first component, scaled:   {first_scaled:.4f} of variance")

rescaled pixel #42
first component, unscaled: 1.0000 of variance
first component, scaled:   0.1203 of variance


Unscaled, the first component absorbs **all** the variance — it *is* that one
rescaled pixel, and the other sixty-three have been squeezed out of the picture
by a change of units. Scaled, it is back to twelve percent. Nothing warned you;
`explained_variance_ratio_` of 1.0000 looks like a triumph.

A detail worth copying: the pixel is chosen by `argmax(var)`. Rescaling pixel 0
instead demonstrates nothing, because in this dataset pixel 0 is a corner that
is always zero — and a thousand times zero is still zero.

## 2. k-means always returns k clusters

Including on data with no clusters at all. Below, the same procedure runs on
three real blobs and on uniform noise; the inertia curve is the only thing that
distinguishes them, and only if you look.

In [4]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

blobs, _ = make_blobs(n_samples=400, centers=3, cluster_std=0.6, random_state=0)
noise = rng.uniform(-8, 8, size=(400, 2))

for name, data in [("3 real blobs", blobs), ("uniform noise", noise)]:
    line = []
    for k in range(2, 7):
        km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(data)
        line.append(f"k={k}: sil={silhouette_score(data, km.labels_):.2f}")
    print(f"{name:<14} " + "  ".join(line))

3 real blobs   k=2: sil=0.55  k=3: sil=0.66  k=4: sil=0.54  k=5: sil=0.42  k=6: sil=0.33
uniform noise  k=2: sil=0.37  k=3: sil=0.38  k=4: sil=0.41  k=5: sil=0.39  k=6: sil=0.40


The blobs peak sharply at `k=3` (0.66) and fall away to 0.33 by `k=6`. The noise
curve is **flat** — it drifts between 0.37 and 0.41 and never rises above the
value it started at, which is the signature of "there is no cluster structure
here".

Note that the noise curve has a nominal maximum, at `k=4`. Reporting *that*
number alone would look like a result. It is the shape of the curve, not its
argmax, that carries the information.

Report the curve, not the single number at your chosen k.

## 3. DBSCAN finds shapes k-means cannot — and refuses to answer

k-means partitions space with straight boundaries, so it cannot represent a
crescent. DBSCAN can, and it also labels points `-1` when they belong to no
dense region, which is the honesty k-means lacks.

In [5]:
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_moons
from sklearn.metrics import adjusted_rand_score

moons, moon_labels = make_moons(n_samples=400, noise=0.06, random_state=0)

km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(moons)
db = DBSCAN(eps=0.25, min_samples=5).fit(moons)

print(f"k-means  ARI={adjusted_rand_score(moon_labels, km.labels_):.3f}")
print(f"DBSCAN   ARI={adjusted_rand_score(moon_labels, db.labels_):.3f}  "
      f"noise points: {(db.labels_ == -1).sum()}")

k-means  ARI=0.274
DBSCAN   ARI=1.000  noise points: 0


`eps` is the parameter that matters and it has no default worth trusting. Too
small and everything is noise; too large and everything is one cluster:

In [6]:
for eps in (0.10, 0.18, 0.25, 0.60):
    labels = DBSCAN(eps=eps, min_samples=5).fit_predict(moons)
    n_clusters = len(set(labels) - {-1})
    print(f"eps={eps:<5} clusters={n_clusters}  noise={(labels == -1).sum():>3}  "
          f"ARI={adjusted_rand_score(moon_labels, labels):.3f}")

eps=0.1   clusters=4  noise= 10  ARI=0.731
eps=0.18  clusters=2  noise=  0  ARI=1.000
eps=0.25  clusters=2  noise=  0  ARI=1.000
eps=0.6   clusters=1  noise=  0  ARI=0.000


## 4. t-SNE: what the picture is allowed to tell you

t-SNE preserves neighbourhoods and discards global geometry. Two consequences
that are easy to state and easy to forget under a pretty plot: **cluster sizes
mean nothing**, and **distances between clusters mean nothing**.

The check that costs one line is to run it twice.

In [7]:
from sklearn.manifold import TSNE

sub = X[:500]
embeddings = [TSNE(n_components=2, perplexity=30, random_state=s,
                   init="pca").fit_transform(sub) for s in (0, 1)]

for s, emb in zip((0, 1), embeddings):
    spread = emb.max(axis=0) - emb.min(axis=0)
    print(f"seed {s}: extent {np.round(spread, 1)}")

print("\nThe two runs are not comparable point-for-point; what should survive")
print("is which points sit together, not where the groups land.")

seed 0: extent [52.2 57.8]
seed 1: extent [52.1 55.8]

The two runs are not comparable point-for-point; what should survive
is which points sit together, not where the groups land.


If a structure appears under one seed and not another, it is a property of the
optimiser, not of your data. Say in the caption which seed produced the figure
you printed.

---

## Using this in the capstone

- **PCA** — report the cumulative variance curve, not just the component count
  you chose. Scale first, always.
- **k-means** — report silhouette across a range of k. A flat curve means no
  cluster structure, and that is a finding worth writing down rather than hiding.
- **DBSCAN** — report `eps` and the noise-point count; a run that labels 60% of
  the data as noise is not a clustering.
- **t-SNE** — two seeds, and a caption naming the one you show.

## Where to go next

- **Reading, Sessions 15–18** — the mechanics behind each of these checks.
- **Session 16's reading** on why t-SNE distances are decoration.